# Proportionality and limits of temporal transfer

Eight new figures. Specification v2 was frozen before the extension fits, after the completed stability results were inspected. Prior populations, exclusions and holdouts are unchanged.

In [ ]:
import sys
# For notebooks inside research/ftir_hips_chem/:
sys.path.insert(0, './scripts')
# For notebooks under notebooks/ (e.g. plotting_gaps_scenarios.ipynb):
# sys.path.insert(0, '../research/ftir_hips_chem/scripts')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Config + data
from config import (
    SITES, PROCESSED_SITES_DIR, FILTER_DATA_PATH,
    AERONET_DATA_DIR, WEATHER_DATA_DIR, MAC_VALUE,
)

# Exclusions (use these — do not hand-filter)
from outliers import (
    EXCLUDED_SAMPLES, MANUAL_OUTLIERS,
    apply_exclusion_flags, apply_threshold_flags,
    get_clean_data, print_exclusion_summary,
)

# Data loading / matching
from data_matching import (
    load_aethalometer_data, load_filter_data,
    match_aeth_filter_data, match_all_parameters,
)
from etad_factors import load_etad_factor_contributions, match_etad_factors

# External datasets — these resolve their own location, so do NOT hardcode a
# Drive path. Each checks an env var, then a config constant, then discovers the
# Drive mount: AETHMODULAR_AERONET_DIR / AETHMODULAR_IMPROVE_DIR.
from aeronet import load_aeronet, aeronet_dir, COLS as AERONET_COLS
from improve_io import load_improve_clean

# Plotting — importing the package auto-applies the white-background default
# style (apply_default_style()). Do NOT call plt.style.use('seaborn-v0_8-darkgrid')
# afterwards — it re-adds the grey axes facecolor we don't want.
from plotting import PlotConfig, crossplots, timeseries, distributions, comparisons
from plotting.utils import calculate_regression_stats

PlotConfig.set(sites='all', layout='individual', show_stats=False, show_1to1=False)


The canonical load/flag/clean procedure was completed in the frozen baseline. This notebook reuses its hashed membership and source links; it does not silently apply new exclusions.

In [ ]:
from pathlib import Path
import importlib.util
from IPython.display import display
path=Path('workflows/analyze_filter_proportionality.py').resolve()
spec=importlib.util.spec_from_file_location('proportional_workflow',path)
workflow=importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)
tables,figure_paths=workflow.run()
b=tables['proportionality_blocks']
s=tables['proportionality_summary']
ib=tables['id11_training_blocks']
cohorts=pd.read_parquet(workflow.FROZEN/'population_membership_summary.parquet')
from plotting import filter_proportionality as charts

## The proportionality question

In [ ]:
fig = charts.paired_blocks(b)
display(fig)
plt.close(fig)

**Notes.** Positive paired MAE differences favor an intercept. Addis improves by 7.64 Mm⁻¹ overall and wins all eight quarters. Other sites show less consistent results; Delhi’s aggregate favors proportional prediction despite three quarter-level intercept wins. Statistical coefficients are not physical MAC estimates.

## Quarter-specific bias

In [ ]:
fig = charts.bias_blocks(b)
display(fig)
plt.close(fig)

**Notes.** Signed errors are predicted minus reported HIPS. Bias persists with either model form. Beijing 2024Q3 remains included and has no test EC beyond its training range. The diagnostic cohort retains all five nonpositive EC predictions; no model predictions were clipped.

## Prediction into later periods

In [ ]:
fig = charts.forward_errors(b)
display(fig)
plt.close(fig)

**Notes.** All three models use the same earlier-only training folds and common test filters within a scheme. Addis’s intercept model wins all six supported quarters. Delhi retains strong negative signed error with either EC-based model. Early unavailable folds remain shown.

## Fixed denominator populations

In [ ]:
fig = charts.denominator_sensitivity(s,cohorts)
display(fig)
plt.close(fig)

**Notes.** Every sensitivity uses saved membership. Counts show full populations; JPL’s seven-filter 2× subset has zero supported folds. Changing populations changes concentration distributions and baseline errors together, so this is not evidence that a method deteriorates at higher EC. No threshold was tuned.

## What receives equal weight

In [ ]:
fig = charts.weighting(s)
display(fig)
plt.close(fig)

**Notes.** Delhi’s primary intercept bias is −1.779 Mm⁻¹ over filters and +2.151 Mm⁻¹ over quarters. Both estimates are retained. Later-period bias remains negative under either weighting. Marker annotations report both choices, not uncertainty bounds.

## Training on earlier ID-11 filters

In [ ]:
fig = charts.id11_errors(ib)
display(fig)
plt.close(fig)

**Notes.** Identical 127 later ID-11 filters are compared under both training choices. OLS MAE falls from 4.009 to 3.884 Mm⁻¹ and mean error from +1.621 to +0.619 Mm⁻¹. This is a bounded predictive sensitivity; ID/date confounding prevents attributing it to a model change. No residual means were subtracted.

## Which metadata folds are supported

In [ ]:
fig = charts.id11_support(ib)
display(fig)
plt.close(fig)

**Notes.** All-earlier and restricted training must each meet the same minimum. The first two ID-11 test quarters are unavailable for a paired comparison. Common support begins in 2023Q3 and covers five quarters; compare these same filters rather than the previous 155-filter forward aggregate.

## Why Delhi’s final quarter matters

In [ ]:
fig = charts.test_weights(b)
display(fig)
plt.close(fig)

**Notes.** Delhi’s 2024Q2 quarter supplies 26 of 38 later-period test filters (68.4%). Filter-weighted errors emphasize that quarter; equal-quarter errors assign each supported quarter the same weight. Neither is chosen for a preferred sign.

## Evidence and limits

The [report](/Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem/output/tables/filter_proportionality/proportionality_temporal_transfer_report.md), [extended unsent questions](/Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem/output/tables/filter_proportionality/upstream_questions_v2_draft.md) and [USPA-0257 evidence package](/Users/ahmadjalil/github/aethmodular/research/ftir_hips_chem/output/tables/filter_proportionality/USPA-0257_evidence_package.md) retain source links and unresolved requirements. Corrected instrument observations can be valid when their history is documented. No verified interval mean, physical offset or upstream FTIR independence is claimed.